# Assignment 5: Ensemble Learning (Bagging, Boosting, Stacking, Voting)
Dataset: Breast Cancer Wisconsin (Diagnostic), built into scikit-learn. Run all cells top to bottom.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, AdaBoostClassifier,
                              StackingClassifier, VotingClassifier)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, confusion_matrix)
sns.set_style("whitegrid")
SEED = 7

## 1. Load data and split (same 80:20 stratified split as Assignment 4)

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target            # 0 = malignant, 1 = benign
class_names = list(data.target_names)
print("Shape:", X.shape, "| missing values:", int(X.isnull().sum().sum()))
print(y.map({0: "malignant", 1: "benign"}).value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 2. Base (individual) classifiers
SVM and KNN use scaling inside a pipeline; tree and Naive Bayes do not need it.

In [ ]:
svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, probability=True, random_state=SEED))
dt  = DecisionTreeClassifier(random_state=SEED)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
nb  = GaussianNB()

base_models = {"SVM": svm, "Decision Tree": dt, "KNN": knn, "Naive Bayes": nb}

## 3. Ensemble models
* **Bagging**: 100 full decision trees on bootstrap samples (variance reduction)
* **Random Forest**: bagging plus random feature subsets
* **AdaBoost**: sequential boosting of shallow trees (depth 1 stumps vs depth 2)
* **Voting**: hard (majority) and soft (average probability)
* **Stacking**: SVM, KNN, Decision Tree, Naive Bayes combined by a Logistic Regression meta-classifier

In [ ]:
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=SEED),
                            n_estimators=100, random_state=SEED)
rf = RandomForestClassifier(n_estimators=200, random_state=SEED)

ada_stump = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                               n_estimators=100, learning_rate=0.5, random_state=SEED)
ada_d2    = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                               n_estimators=100, learning_rate=0.5, random_state=SEED)

estimators = [("svm", svm), ("dt", dt), ("knn", knn), ("nb", nb)]
vote_hard = VotingClassifier(estimators, voting="hard")
vote_soft = VotingClassifier(estimators, voting="soft")

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
stack = StackingClassifier(estimators=estimators,
                           final_estimator=LogisticRegression(max_iter=1000),
                           cv=cv5, passthrough=False)

ensemble_models = {"Bagging (DT)": bagging, "Random Forest": rf,
                   "AdaBoost (depth 1)": ada_stump, "AdaBoost (depth 2)": ada_d2,
                   "Voting (hard)": vote_hard, "Voting (soft)": vote_soft,
                   "Stacking (LR meta)": stack}
all_models = {**base_models, **ensemble_models}

## 4. Train and evaluate on the test set

In [ ]:
rows, preds = [], {}
for name, m in all_models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test); preds[name] = p
    row = {"Model": name,
           "Type": "Base" if name in base_models else "Ensemble",
           "Accuracy": accuracy_score(y_test, p),
           "Precision": precision_score(y_test, p),
           "Recall": recall_score(y_test, p),
           "F1-Score": f1_score(y_test, p)}
    if hasattr(m, "predict_proba") and name != "Voting (hard)":
        pr = m.predict_proba(X_test)
        row["ROC-AUC"] = roc_auc_score(y_test, pr[:, 1])
        row["Log Loss"] = log_loss(y_test, pr)
    rows.append(row)

results = pd.DataFrame(rows).set_index("Model").round(4)
display(results)
results.to_csv("results_ens.csv")

## 5. Cross-validation (5-fold x 3 repeats on the whole dataset)
A single 114-sample test set is small, so repeated CV gives a more reliable ranking.

In [ ]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
cv_scores = {n: cross_val_score(m, X, y, cv=rcv, scoring="accuracy", n_jobs=-1)
             for n, m in all_models.items()}
cv_table = pd.DataFrame({"CV Acc mean": {n: s.mean() for n, s in cv_scores.items()},
                         "CV Acc std":  {n: s.std()  for n, s in cv_scores.items()}}).round(4)
display(cv_table.sort_values("CV Acc mean", ascending=False))
cv_table.to_csv("cv_table.csv")

## 6. Charts

In [ ]:
order = results.sort_values("Accuracy").index
colors = ["#4C72B0" if results.loc[n, "Type"] == "Base" else "#DD8452" for n in order]
plt.figure(figsize=(7, 4.5))
plt.barh(order, results.loc[order, "Accuracy"], color=colors)
for i, v in enumerate(results.loc[order, "Accuracy"]): plt.text(v + .001, i, f"{v:.3f}", va="center", fontsize=8)
plt.xlim(0.88, 1.0); plt.xlabel("Test accuracy"); plt.title("Base (blue) vs ensemble (orange) models")
plt.tight_layout(); plt.savefig("fig_acc.png", dpi=200); plt.show()

plt.figure(figsize=(7.5, 4.2))
names = list(cv_scores)
plt.boxplot([cv_scores[n] for n in names])
plt.xticks(range(1, len(names) + 1), names, rotation=45, ha="right"); plt.ylabel("CV accuracy (15 folds)")
plt.title("Repeated CV accuracy distribution"); plt.tight_layout()
plt.savefig("fig_cv.png", dpi=200); plt.show()

sel = ["Decision Tree", "Bagging (DT)", "AdaBoost (depth 1)", "Stacking (LR meta)"]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, n in zip(axes, sel):
    sns.heatmap(confusion_matrix(y_test, preds[n]), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(n, fontsize=9); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.savefig("fig_cm.png", dpi=200); plt.show()

## 7. Effect of the number of estimators (bagging vs boosting)

In [ ]:
ns = [1, 5, 10, 25, 50, 100, 200]
bag_s, ada_s = [], []
for n in ns:
    b = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=SEED), n_estimators=n, random_state=SEED)
    a = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=n, learning_rate=0.5, random_state=SEED)
    bag_s.append(cross_val_score(b, X, y, cv=cv5).mean())
    ada_s.append(cross_val_score(a, X, y, cv=cv5).mean())
plt.figure(figsize=(5.5, 3.8))
plt.plot(ns, bag_s, "o-", label="Bagging (deep trees)")
plt.plot(ns, ada_s, "s-", label="AdaBoost (stumps)")
plt.xscale("log"); plt.xlabel("Number of estimators"); plt.ylabel("5-fold CV accuracy")
plt.legend(); plt.title("Accuracy vs number of estimators"); plt.tight_layout()
plt.savefig("fig_nest.png", dpi=200); plt.show()
print(dict(zip(ns, np.round(bag_s, 4)))); print(dict(zip(ns, np.round(ada_s, 4))))

## 8. Stacking meta-learner weights and error summary

In [ ]:
meta = stack.final_estimator_
print("Meta-classifier (Logistic Regression) coefficients on base-model probabilities:")
print(dict(zip(["SVM", "DecisionTree", "KNN", "NaiveBayes"], np.round(meta.coef_[0], 3))))
for n in ["Decision Tree", "Bagging (DT)", "Random Forest", "AdaBoost (depth 1)", "AdaBoost (depth 2)",
          "Voting (hard)", "Voting (soft)", "Stacking (LR meta)", "SVM", "KNN"]:
    cm = confusion_matrix(y_test, preds[n])
    print(f"{n:20s} malignant->benign (FN for cancer) = {cm[0,1]} | benign->malignant = {cm[1,0]}")